# Lab 02 — Chapter 18: Bài tập Perplexity

Notebook này trình bày phần lý thuyết và phân tích cho bài tập perplexity. Không có code cell.

## 1. Ý nghĩa của perplexity

Với chuỗi từ $W = w_1, \ldots, w_N$, perplexity được định nghĩa là:

$$\operatorname{PPL}(W) = P(W)^{-1/N} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N}\log P(w_i \mid w_{<i})\right).$$

Nó có thể được hiểu là số lựa chọn trung bình mà mô hình còn phân vân ở mỗi vị trí từ. Perplexity càng thấp càng tốt, vì mô hình gán xác suất cao hơn cho dữ liệu thực tế. Chỉ so sánh PPL khi dùng cùng corpus đánh giá, cùng cách tách token, cùng từ vựng và cùng quy tắc xử lý ký hiệu đầu/cuối câu.

## 2. Đánh giá đúng quy trình

- **Train set** dùng để ước lượng các count và xác suất N-gram. PPL trên train thường lạc quan, nên không đủ để kết luận mô hình tốt.
- **Validation set** dùng để chọn bậc N-gram và cách smoothing. Đây là nơi so sánh MLE với Laplace hoặc các cấu hình khác.
- **Test set** chỉ dùng sau khi chốt cấu hình. Kết quả test là ước lượng khả năng khái quát hóa cuối cùng.

Do đó, không nên chọn mô hình dựa trên PPL test. Làm vậy khiến thông tin test bị rò rỉ vào quá trình thiết kế mô hình.

## 3. MLE, Laplace và perplexity

MLE gán $P(w\mid h)=c(h,w)/c(h)$. Nếu một N-gram xuất hiện ở validation hoặc test nhưng chưa từng xuất hiện trong train, xác suất bằng 0; xác suất của cả câu bằng 0 và PPL trở thành vô hạn. Vì vậy MLE thường có PPL train tốt nhưng không ổn định trên dữ liệu chưa thấy.

Laplace smoothing dùng $P(w\mid h)=[c(h,w)+1]/[c(h)+|V|]$. Cách này dành một phần xác suất cho các từ hoặc N-gram chưa thấy, nên PPL trên validation/test luôn hữu hạn nếu token thuộc vocabulary. Đổi lại, Laplace có thể phân bổ quá nhiều xác suất cho sự kiện hiếm; vì thế phải chọn theo PPL validation thay vì mặc định coi Laplace luôn tốt hơn.

## 4. Xác suất câu và xếp hạng câu

Xác suất câu được tính theo quy tắc dây chuyền: $P(w_1,\ldots,w_N)=\prod_i P(w_i\mid w_{<i})$. Khi so sánh các câu tiếp nối cùng một context, câu có tổng log-probability lớn hơn (ít âm hơn) được xếp hạng cao hơn. Dùng log-probability giúp tránh numerical underflow khi nhân nhiều xác suất nhỏ.

Một continuation tự nhiên, phù hợp với context và các N-gram đã quan sát thường có điểm cao hơn continuation chứa tổ hợp từ vô nghĩa. Tuy nhiên, điểm số chỉ phản ánh phân bố của corpus train; nó không tự bảo đảm tính đúng sự thật, ngữ pháp đầy đủ hoặc chất lượng ngữ nghĩa sâu.

## 5. Unseen words và zero probability

Một từ chưa có trong vocabulary train là **out-of-vocabulary (OOV)**. Nếu giữ nguyên từ đó trong mô hình MLE, xác suất có thể bằng 0 và làm PPL vô hạn. Cách xử lý nhất quán là xây dựng vocabulary chỉ từ train, thay các từ hiếm/OOV ở validation và test bằng token `<UNK>`, rồi học xác suất cho `<UNK>` từ train.

Không được mở rộng vocabulary bằng dữ liệu test, vì đó là data leakage. Mô hình neural như RNN hoặc Transformer thường giảm vấn đề OOV nhờ tokenization subword, nhưng vẫn cần đánh giá trên test tách biệt và vẫn có thể sai với từ hiếm hoặc domain khác.

## 6. Error analysis

Các nguyên nhân chính khiến PPL validation/test cao gồm:

1. **Data sparsity:** bigram/trigram cụ thể xuất hiện quá ít trong train.
2. **OOV hoặc unseen N-gram:** MLE gán xác suất 0; smoothing hoặc `<UNK>` chưa được áp dụng đúng.
3. **Khác biệt domain:** train và test có chủ đề, phong cách hoặc từ vựng khác nhau.
4. **Tiền xử lý không nhất quán:** khác tokenization, chữ hoa/thường, số, dấu câu hoặc token boundary.
5. **Context dài quá thưa:** tăng bậc N-gram làm mô hình ghi nhớ train tốt hơn nhưng có thể tổng quát hóa kém hơn.

Cách khắc phục phù hợp là kiểm tra tokenization, dùng `<UNK>`, áp dụng smoothing, điều chỉnh bậc N-gram theo validation và bổ sung dữ liệu cùng domain.

## 7. Kết luận

PPL là thước đo nội tại hữu ích để so sánh các language model trong cùng điều kiện đánh giá. Mô hình được chọn cần có PPL validation thấp, sau đó báo cáo duy nhất một kết quả trên test. Khi diễn giải kết quả, cần nêu rõ bậc N-gram, smoothing, vocabulary/OOV policy, cách chia dữ liệu và tokenization để phép so sánh có ý nghĩa và tái lập được.